In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import load_model
import joblib

loaded_model = load_model('model_bigru.keras')
scaler = joblib.load('scaler_minmax_gru.pkl')

def create_sequences(data, sequence_length):
    X, y = [], []
    for i in range(len(data) - sequence_length):
        X.append(data[i:i+sequence_length])
        y.append(data[i+sequence_length])
    return np.array(X), np.array(y)

def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    return np.mean(numerator / denominator) * 100

data_folder = 'stock/'
sequence_length = 3
prix = {}
dates = {}  # Nouveau dictionnaire pour stocker les dates
metrics_dict = {}
action_files = [f for f in os.listdir(data_folder) if f.endswith('.csv')]
for file in action_files:
    action_name = os.path.splitext(file)[0]
    
    # Charger les données avec l'index des dates
    data = pd.read_csv(os.path.join(data_folder, file), index_col='Date', parse_dates=True)['Close']
    
    # Normaliser les données
    scaled_data = scaler.transform(data.values.reshape(-1, 1))
    
    # Créer des séquences
    X, y = create_sequences(scaled_data, sequence_length)
    X = X.reshape((X.shape[0], X.shape[1], 1))
    
    # Faire des prédictions
    predictions = loaded_model.predict(X)
    
    # Revenir à l'échelle originale
    predictions_unscaled = scaler.inverse_transform(predictions)
    y_unscaled = scaler.inverse_transform(y.reshape(-1, 1))
    
    # Stocker les prix prédits avec les dates correspondantes
    # Les dates commencent après la séquence initiale
    dates_array = data.index[sequence_length:]
    prix[action_name] = predictions_unscaled.flatten()
    dates[action_name] = dates_array
    
    # Calcul des métriques
    metrics_dict[action_name] = {
        'MSE': mean_squared_error(y_unscaled, predictions_unscaled),
        'MAE': mean_absolute_error(y_unscaled, predictions_unscaled),
        'R²': r2_score(y_unscaled, predictions_unscaled),
        'SMAPE': smape(y_unscaled, predictions_unscaled)
    }

# Création du DataFrame final avec les dates
prix_predits = pd.DataFrame()
for action_name in prix:
    prix_predits[action_name] = pd.Series(prix[action_name], index=dates[action_name])

# Sauvegarder le DataFrame avec les dates
prix_predits.to_csv('predictions/prix_bigru.csv')

79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
